In [0]:
# Widgets for catalog parameter
dbutils.widgets.text("catalog", "TPCA")

# Widget for job run id - filled by the job task parameter (job_run_id = {{job.run_id}})
# when this notebook runs as a job task; empty in interactive runs
dbutils.widgets.text("job_run_id", "")

catalog = dbutils.widgets.get("catalog")

In [0]:
# Create the catalog if it does not exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")

# Create the schemas: landing, bronze, silver, gold, and ops
schemas = ["bronze", "silver", "gold"]

for schema in schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")


In [0]:
# Create raw volume inside the landing schema and subfolders
volume_name1 = "raw"
# volume_name2 = "pipeline_state"
full_volume1 = f"{catalog}.bronze.{volume_name1}"
# full_volume2 = f"{catalog}.ops.{volume_name2}"

# Create the volume if it does not exist
spark.sql(f"CREATE VOLUME IF NOT EXISTS {full_volume1}")

# Create the volume if it does not exist
# spark.sql(f"CREATE VOLUME IF NOT EXISTS {full_volume2}")

# Define the folders to create inside the volume
folders = ["airline", "airline_route", "booking", "flight_booking", "nation","passenger" ,"region","route"]

volume_path = f"/Volumes/{catalog}/bronze/{volume_name1}"

for folder in folders:
    folder_path = f"{volume_path}/{folder}"
    try:
        dbutils.fs.ls(folder_path)
    except Exception:
        dbutils.fs.mkdirs(folder_path)
   

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql import DataFrame


def get_job_run_id():
    """Return the current job run id when running as a job task, otherwise None.

    The job task parameter 'job_run_id' is set with the dynamic value reference
    {{job.run_id}}, which Databricks substitutes with the actual job run id.
    In interactive runs the widget stays empty, so None is returned.
    """
    try:
        value = dbutils.widgets.get("job_run_id")
        if value and not value.strip().startswith("{{"):
            return value
    except Exception:
        pass
    return None

def add_metadata(df: DataFrame, source_file: str = None) -> DataFrame:
    """Add _source_file and _ingested_at metadata columns to a DataFrame.
    
    Args:
        df: Input Spark DataFrame.
        source_file: Optional explicit source file path. If None, uses _metadata.file_path (Auto Loader).
    
    Returns:
        DataFrame with _source_file and _ingested_at columns appended.
    """
    if source_file is not None:
        df = df.withColumn("_source_file", lit(source_file))
    else:
        df = df.withColumn("_source_file", col("_metadata.file_path"))
    
    df = df.withColumn("_ingested_at", current_timestamp())
    df = df.withColumn("pipeline_run_id", lit(get_job_run_id()).cast("string"))
    return df